In [84]:
import os

print(os.getcwd())

/content


In [85]:
from pathlib import Path

for p in Path(".").iterdir():
    print(p)

.config
drive
sample_data


In [86]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [87]:
BASE_DIR = "/content/drive/MyDrive/Delhivery_Graph_ETA"

REPORTS_DIR = f"{BASE_DIR}/reports"
MODELS_DIR = f"{BASE_DIR}/models"
GRAPH_DIR = f"{BASE_DIR}/data/graph_data"
FEATURE_DIR = f"{BASE_DIR}/data/feature_store"

In [88]:
import pandas as pd

source_risk = pd.read_csv(
    f"{REPORTS_DIR}/phase7_top_source_facilities.csv"
)

destination_risk = pd.read_csv(
    f"{REPORTS_DIR}/phase7_top_destination_facilities.csv"
)

corridor_risk = pd.read_csv(
    f"{REPORTS_DIR}/phase7_high_risk_corridors.csv"
)

source_risk.head()

,source_center,shipments,avg_actual_factor,avg_pred_factor,impact_score
0,IND000000ACB,4836,1.925056,1.885241,9117.025903
1,IND562132AAA,2005,1.817903,1.826613,3662.359147
2,IND421302AAG,1643,2.120249,2.120747,3484.387296
3,IND411033AAA,678,1.920938,2.012705,1364.613721
4,IND501359AAE,678,1.954012,2.009997,1362.778249


In [89]:
import os
import json
import joblib
import shap

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit

In [90]:
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Delhivery_Graph_ETA"
REPORTS_DIR = f"{BASE_DIR}/reports"

files = [

    "phase7_top_source_facilities.csv",
    "phase7_top_destination_facilities.csv",
    "phase7_high_risk_corridors.csv",
    "phase7_top_impact_corridors.csv",
    "phase7_graph_contribution_breakdown.csv",
    "phase7_global_shap_importance.csv",
    "phase7_executive_findings.csv"

]

for f in files:

    print("\n")
    print("=" * 100)
    print(f)

    df = pd.read_csv(f"{REPORTS_DIR}/{f}")

    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())

    print("\nhead:")
    print(df.head(3))



phase7_top_source_facilities.csv
shape: (20, 5)
columns:
['source_center', 'shipments', 'avg_actual_factor', 'avg_pred_factor', 'impact_score']

head:
  source_center  shipments  avg_actual_factor  avg_pred_factor  impact_score
0  IND000000ACB       4836           1.925056         1.885241   9117.025903
1  IND562132AAA       2005           1.817903         1.826613   3662.359147
2  IND421302AAG       1643           2.120249         2.120747   3484.387296


phase7_top_destination_facilities.csv
shape: (20, 5)
columns:
['destination_center', 'shipments', 'avg_actual_factor', 'avg_pred_factor', 'impact_score']

head:
  destination_center  shipments  avg_actual_factor  avg_pred_factor  \
0       IND000000ACB       2512           1.893871         1.873031   
1       IND562132AAA       1993           1.765580         1.731575   
2       IND421302AAG       1241           2.226164         2.343425   

   impact_score  
0   4705.054674  
1   3451.029305  
2   2908.189927  


phase7_high_risk_

In [91]:
import pandas as pd
import json

BASE_DIR = "/content/drive/MyDrive/Delhivery_Graph_ETA"

REPORTS_DIR = f"{BASE_DIR}/reports"
MODELS_DIR = f"{BASE_DIR}/models"

source_risk = pd.read_csv(
    f"{REPORTS_DIR}/phase7_top_source_facilities.csv"
)

destination_risk = pd.read_csv(
    f"{REPORTS_DIR}/phase7_top_destination_facilities.csv"
)

corridor_risk = pd.read_csv(
    f"{REPORTS_DIR}/phase7_high_risk_corridors.csv"
)

impact_corridors = pd.read_csv(
    f"{REPORTS_DIR}/phase7_top_impact_corridors.csv"
)

graph_breakdown = pd.read_csv(
    f"{REPORTS_DIR}/phase7_graph_contribution_breakdown.csv"
)

shap_imp = pd.read_csv(
    f"{REPORTS_DIR}/phase7_global_shap_importance.csv"
)

executive_findings = pd.read_csv(
    f"{REPORTS_DIR}/phase7_executive_findings.csv"
)

with open(
    f"{MODELS_DIR}/lgbm_final_graph_edge_metrics.json"
) as f:
    metrics = json.load(f)

In [92]:
executive_summary = pd.DataFrame({

    "Metric":[
        "Facilities",
        "Corridors",
        "Communities",
        "Final MAE",
        "Final RMSE",
        "Final R2"
    ],

    "Value":[
        1657,
        2783,
        94,
        metrics["MAE"],
        metrics["RMSE"],
        metrics["R2"]
    ]
})

executive_summary

,Metric,Value
0,Facilities,1657.000000
1,Corridors,2783.000000
2,Communities,94.000000
3,Final MAE,0.405637
4,Final RMSE,0.936562
5,Final R2,0.636988


In [93]:
graph_breakdown

graph_breakdown.sort_values(
    "Graph_%",
    ascending=False
)

,Component,Importance,Graph_%
0,Embeddings,0.718120,65.237402
1,Centrality,0.182564,16.585002
2,Edge Intelligence,0.142387,12.935136
3,Community,0.057708,5.242460


In [94]:
source_risk.sort_values(
    "impact_score",
    ascending=False
).head(10)

,source_center,shipments,avg_actual_factor,avg_pred_factor,impact_score
0,IND000000ACB,4836,1.925056,1.885241,9117.025903
1,IND562132AAA,2005,1.817903,1.826613,3662.359147
2,IND421302AAG,1643,2.120249,2.120747,3484.387296
3,IND411033AAA,678,1.920938,2.012705,1364.613721
4,IND501359AAE,678,1.954012,2.009997,1362.778249
5,IND712311AAA,477,2.195399,2.283553,1089.254600
6,IND395023AAA,545,1.911993,1.868178,1018.157239
7,IND110037AAM,469,2.253018,2.118428,993.542934
8,IND160002AAC,422,1.899615,1.886956,796.295571
9,IND751002AAB,296,2.008521,2.199491,651.049323


In [95]:
destination_risk.sort_values(
    "impact_score",
    ascending=False
).head(10)

,destination_center,shipments,avg_actual_factor,avg_pred_factor,impact_score
0,IND000000ACB,2512,1.893871,1.873031,4705.054674
1,IND562132AAA,1993,1.765580,1.731575,3451.029305
2,IND421302AAG,1241,2.226164,2.343425,2908.189927
3,IND712311AAA,1092,2.442546,2.441265,2665.861545
4,IND501359AAE,1126,1.934613,1.967197,2215.064008
5,IND411033AAA,709,2.076402,2.059124,1459.918864
6,IND462022AAA,650,2.130733,2.212693,1438.250141
7,IND131028AAB,659,1.869114,1.901896,1253.349374
8,IND110037AAM,493,2.025727,2.085853,1028.325746
9,IND160002AAC,505,2.004790,1.978636,999.211273


In [96]:
impact_corridors.sort_values(
    "impact_score",
    ascending=False
)

,corridor,shipments,avg_actual_factor,avg_pred_factor,impact_score
0,IND000000ACB -> IND562132AAA,1070,1.744211,1.688481,1806.674239
1,IND000000ACB -> IND712311AAA,597,2.435245,2.160609,1289.883763
2,IND000000ACB -> IND501359AAE,443,1.925686,1.828349,809.958484
3,IND562132AAA -> IND000000ACB,460,1.785745,1.698787,781.441933
4,IND000000ACB -> IND421302AAG,391,1.897166,1.991502,778.677256
5,IND421302AAG -> IND131028AAB,318,1.853458,1.923782,611.762614
6,IND501359AAE -> IND462022AAA,214,2.039759,2.240027,479.365784
7,IND395023AAA -> IND110037AAM,258,1.794088,1.843195,475.544295
8,IND421302AAG -> IND712311AAA,220,1.945452,2.158974,474.974174
9,IND562132AAA -> IND462022AAA,208,2.328937,2.198276,457.241445


In [97]:
corridor_priority = impact_corridors.copy()

corridor_priority["priority"] = [
    "Critical" if i < 5
    else "High"
    for i in range(len(corridor_priority))
]

In [98]:
top20 = shap_imp.head(20)

top20

,feature,mean_abs_shap
0,num__trip_to_cutoff_minutes,0.149940
1,num__osrm_speed,0.124493
2,cat__route_type_Carting,0.075812
3,num__trip_hour,0.058914
4,num__betweenness_difference,0.054667
5,num__trip_to_scan_minutes,0.046648
6,num__source_volume,0.045459
7,num__corridor_volume,0.041696
8,num__embedding_cosine_similarity,0.037519
9,num__dst_embedding_6,0.031485


In [99]:
ftl_carting_framework = pd.DataFrame({

    "Scenario":[
        "High-volume corridor",
        "Backbone hub corridor",
        "Long-haul corridor",
        "High-frequency corridor",
        "Low-volume regional corridor",
        "Flexible last-mile corridor",
        "Demand-uncertain corridor"
    ],

    "Recommended_Route_Type":[
        "FTL",
        "FTL",
        "FTL",
        "FTL",
        "Carting",
        "Carting",
        "Carting"
    ],

    "Reason":[
        "Better vehicle utilization",
        "Protect critical network links",
        "Lower per-unit transport cost",
        "Stable predictable demand",
        "Avoid under-utilized trucks",
        "Greater operational flexibility",
        "Lower fixed capacity commitment"
    ]
})

ftl_carting_framework

,Scenario,Recommended_Route_Type,Reason
0,High-volume corridor,FTL,Better vehicle utilization
1,Backbone hub corridor,FTL,Protect critical network links
2,Long-haul corridor,FTL,Lower per-unit transport cost
3,High-frequency corridor,FTL,Stable predictable demand
4,Low-volume regional corridor,Carting,Avoid under-utilized trucks
5,Flexible last-mile corridor,Carting,Greater operational flexibility
6,Demand-uncertain corridor,Carting,Lower fixed capacity commitment


FTL should be prioritized on high-volume,
high-centrality network corridors where
capacity utilization remains consistently high.

Carting should be prioritized on lower-volume,
regional, or flexible corridors where shipment
demand is more variable.

Route type emerged as one of the strongest
ETA drivers in SHAP analysis, indicating that
routing strategy materially impacts delivery
performance.

In [100]:
import numpy as np

hub_priority = source_risk.copy()

q80 = hub_priority["impact_score"].quantile(0.80)
q60 = hub_priority["impact_score"].quantile(0.60)

hub_priority["Priority"] = np.where(
    hub_priority["impact_score"] >= q80,
    "Tier 1",
    np.where(
        hub_priority["impact_score"] >= q60,
        "Tier 2",
        "Tier 3"
    )
)

hub_priority = hub_priority.sort_values(
    "impact_score",
    ascending=False
)

hub_priority

,source_center,shipments,avg_actual_factor,avg_pred_factor,impact_score,Priority
0,IND000000ACB,4836,1.925056,1.885241,9117.025903,Tier 1
1,IND562132AAA,2005,1.817903,1.826613,3662.359147,Tier 1
2,IND421302AAG,1643,2.120249,2.120747,3484.387296,Tier 1
3,IND411033AAA,678,1.920938,2.012705,1364.613721,Tier 1
4,IND501359AAE,678,1.954012,2.009997,1362.778249,Tier 2
5,IND712311AAA,477,2.195399,2.283553,1089.254600,Tier 2
6,IND395023AAA,545,1.911993,1.868178,1018.157239,Tier 2
7,IND110037AAM,469,2.253018,2.118428,993.542934,Tier 2
8,IND160002AAC,422,1.899615,1.886956,796.295571,Tier 3
9,IND751002AAB,296,2.008521,2.199491,651.049323,Tier 3


In [101]:
hub_upgrade_candidates = hub_priority[[
    "source_center",
    "shipments",
    "impact_score",
    "Priority"
]]

hub_upgrade_candidates

,source_center,shipments,impact_score,Priority
0,IND000000ACB,4836,9117.025903,Tier 1
1,IND562132AAA,2005,3662.359147,Tier 1
2,IND421302AAG,1643,3484.387296,Tier 1
3,IND411033AAA,678,1364.613721,Tier 1
4,IND501359AAE,678,1362.778249,Tier 2
5,IND712311AAA,477,1089.254600,Tier 2
6,IND395023AAA,545,1018.157239,Tier 2
7,IND110037AAM,469,993.542934,Tier 2
8,IND160002AAC,422,796.295571,Tier 3
9,IND751002AAB,296,651.049323,Tier 3


In [102]:
for _, row in hub_upgrade_candidates.head(5).iterrows():

    print(
        f"""
Facility: {row['source_center']}

Priority: {row['Priority']}

Shipments: {row['shipments']}

Impact Score: {row['impact_score']:.2f}

Recommended Action:
- Capacity review
- Staffing review
- Sortation review
- Contingency planning
"""
    )


Facility: IND000000ACB

Priority: Tier 1

Shipments: 4836

Impact Score: 9117.03

Recommended Action:
- Capacity review
- Staffing review
- Sortation review
- Contingency planning


Facility: IND562132AAA

Priority: Tier 1

Shipments: 2005

Impact Score: 3662.36

Recommended Action:
- Capacity review
- Staffing review
- Sortation review
- Contingency planning


Facility: IND421302AAG

Priority: Tier 1

Shipments: 1643

Impact Score: 3484.39

Recommended Action:
- Capacity review
- Staffing review
- Sortation review
- Contingency planning


Facility: IND411033AAA

Priority: Tier 1

Shipments: 678

Impact Score: 1364.61

Recommended Action:
- Capacity review
- Staffing review
- Sortation review
- Contingency planning


Facility: IND501359AAE

Priority: Tier 2

Shipments: 678

Impact Score: 1362.78

Recommended Action:
- Capacity review
- Staffing review
- Sortation review
- Contingency planning



In [103]:
corridor_actions = impact_corridors.copy()

q80 = corridor_actions["impact_score"].quantile(0.80)
q60 = corridor_actions["impact_score"].quantile(0.60)

corridor_actions["Priority"] = np.where(
    corridor_actions["impact_score"] >= q80,
    "Critical",
    np.where(
        corridor_actions["impact_score"] >= q60,
        "High",
        "Medium"
    )
)

corridor_actions = corridor_actions.sort_values(
    "impact_score",
    ascending=False
)

corridor_actions

,corridor,shipments,avg_actual_factor,avg_pred_factor,impact_score,Priority
0,IND000000ACB -> IND562132AAA,1070,1.744211,1.688481,1806.674239,Critical
1,IND000000ACB -> IND712311AAA,597,2.435245,2.160609,1289.883763,Critical
2,IND000000ACB -> IND501359AAE,443,1.925686,1.828349,809.958484,Critical
3,IND562132AAA -> IND000000ACB,460,1.785745,1.698787,781.441933,Critical
4,IND000000ACB -> IND421302AAG,391,1.897166,1.991502,778.677256,High
5,IND421302AAG -> IND131028AAB,318,1.853458,1.923782,611.762614,High
6,IND501359AAE -> IND462022AAA,214,2.039759,2.240027,479.365784,High
7,IND395023AAA -> IND110037AAM,258,1.794088,1.843195,475.544295,High
8,IND421302AAG -> IND712311AAA,220,1.945452,2.158974,474.974174,Medium
9,IND562132AAA -> IND462022AAA,208,2.328937,2.198276,457.241445,Medium


In [104]:
corridor_actions["Recommended_Action"] = corridor_actions[
    "Priority"
].map({

    "Critical":
    "Immediate operational review and routing optimization",

    "High":
    "Capacity balancing and monitoring",

    "Medium":
    "Periodic performance review"
})

corridor_actions

,corridor,shipments,avg_actual_factor,avg_pred_factor,impact_score,Priority,Recommended_Action
0,IND000000ACB -> IND562132AAA,1070,1.744211,1.688481,1806.674239,Critical,Immediate operational review and routing optim...
1,IND000000ACB -> IND712311AAA,597,2.435245,2.160609,1289.883763,Critical,Immediate operational review and routing optim...
2,IND000000ACB -> IND501359AAE,443,1.925686,1.828349,809.958484,Critical,Immediate operational review and routing optim...
3,IND562132AAA -> IND000000ACB,460,1.785745,1.698787,781.441933,Critical,Immediate operational review and routing optim...
4,IND000000ACB -> IND421302AAG,391,1.897166,1.991502,778.677256,High,Capacity balancing and monitoring
5,IND421302AAG -> IND131028AAB,318,1.853458,1.923782,611.762614,High,Capacity balancing and monitoring
6,IND501359AAE -> IND462022AAA,214,2.039759,2.240027,479.365784,High,Capacity balancing and monitoring
7,IND395023AAA -> IND110037AAM,258,1.794088,1.843195,475.544295,High,Capacity balancing and monitoring
8,IND421302AAG -> IND712311AAA,220,1.945452,2.158974,474.974174,Medium,Periodic performance review
9,IND562132AAA -> IND462022AAA,208,2.328937,2.198276,457.241445,Medium,Periodic performance review


In [105]:
eta_actions = pd.DataFrame({

    "Driver":[

        "trip_to_cutoff_minutes",

        "trip_to_scan_minutes",

        "source_volume",

        "corridor_volume",

        "route_type",

        "betweenness_difference",

        "embedding_cosine_similarity"

    ],

    "Business_Action":[

        "Reduce dispatch waiting time",

        "Improve scan turnaround",

        "Facility capacity planning",

        "Corridor load balancing",

        "Route-type optimization",

        "Monitor bottleneck corridors",

        "Graph-aware routing"

    ],

    "Expected_Benefit":[

        "Lower ETA inflation",

        "Faster processing",

        "Reduced hub congestion",

        "Lower corridor delays",

        "Improved routing efficiency",

        "Lower network bottlenecks",

        "Smarter route selection"
    ]
})

eta_actions

,Driver,Business_Action,Expected_Benefit
0,trip_to_cutoff_minutes,Reduce dispatch waiting time,Lower ETA inflation
1,trip_to_scan_minutes,Improve scan turnaround,Faster processing
2,source_volume,Facility capacity planning,Reduced hub congestion
3,corridor_volume,Corridor load balancing,Lower corridor delays
4,route_type,Route-type optimization,Improved routing efficiency
5,betweenness_difference,Monitor bottleneck corridors,Lower network bottlenecks
6,embedding_cosine_similarity,Graph-aware routing,Smarter route selection


In [106]:
roi_framework = pd.DataFrame({

    "Metric":[

        "Annual Shipments",

        "Current Late Delivery Rate",

        "Target Late Delivery Reduction",

        "Cost per Late Delivery",

        "Annual Savings"

    ],

    "Formula":[

        "Business Input",

        "Business Input",

        "Business Input",

        "Business Input",

        """
Annual Shipments
×
Late Delivery Reduction
×
Cost per Late Delivery
"""
    ]
})

roi_framework

,Metric,Formula
0,Annual Shipments,Business Input
1,Current Late Delivery Rate,Business Input
2,Target Late Delivery Reduction,Business Input
3,Cost per Late Delivery,Business Input
4,Annual Savings,\nAnnual Shipments\n×\nLate Delivery Reduction...


In [107]:
annual_shipments = 1_000_000

late_delivery_reduction = 0.05

cost_per_late_delivery = 50

estimated_savings = (

    annual_shipments
    *
    late_delivery_reduction
    *
    cost_per_late_delivery

)

print(
    f"Illustrative Annual Savings = ₹{estimated_savings:,.0f}"
)

Illustrative Annual Savings = ₹2,500,000


Illustrative only.

Actual ROI depends on
shipment volume,
SLA penalties,
customer churn impact,
and operational costs.

In [108]:
deployment_roadmap = pd.DataFrame({

    "Phase":[

        "Current",

        "Stage 1",

        "Stage 2",

        "Stage 3"

    ],

    "Description":[

        "Offline ETA scoring",

        "Batch ETA predictions",

        "Near real-time ETA scoring",

        "Graph-aware operational intelligence platform"

    ]
})

deployment_roadmap

,Phase,Description
0,Current,Offline ETA scoring
1,Stage 1,Batch ETA predictions
2,Stage 2,Near real-time ETA scoring
3,Stage 3,Graph-aware operational intelligence platform


In [109]:
future_roadmap = pd.DataFrame({

    "Priority":[

        1,
        2,
        3,
        4

    ],

    "Research_Direction":[

        "Weighted Node2Vec",

        "Delay-Aware Node2Vec",

        "Dual Embedding Architecture",

        "Graph Neural Networks"
    ],

    "Status":[

        "Future Work",

        "Future Work",

        "Future Work",

        "Future Work"
    ]
})

future_roadmap

,Priority,Research_Direction,Status
0,1,Weighted Node2Vec,Future Work
1,2,Delay-Aware Node2Vec,Future Work
2,3,Dual Embedding Architecture,Future Work
3,4,Graph Neural Networks,Future Work


In [110]:
EXPORT_DIR = f"{BASE_DIR}/reports"

hub_upgrade_candidates.to_csv(
    f"{EXPORT_DIR}/phase8_hub_priorities.csv",
    index=False
)

corridor_actions.to_csv(
    f"{EXPORT_DIR}/phase8_corridor_priorities.csv",
    index=False
)

eta_actions.to_csv(
    f"{EXPORT_DIR}/phase8_eta_reduction_actions.csv",
    index=False
)

deployment_roadmap.to_csv(
    f"{EXPORT_DIR}/phase8_deployment_roadmap.csv",
    index=False
)

future_roadmap.to_csv(
    f"{EXPORT_DIR}/phase8_future_roadmap.csv",
    index=False
)

print("Phase 8 executive outputs exported.")

Phase 8 executive outputs exported.


# Final Operations Strategy Memo

## Executive Summary

Graph intelligence improved ETA prediction MAE from 0.4446 to 0.4056,
representing an 8.62% improvement over traditional logistics features.

The analysis identified several critical facilities and corridors that
contribute disproportionately to network-wide delivery delays.

IND000000ACB emerged as the dominant network hub and highest-impact
facility, indicating a significant operational dependency.

## Priority Actions

### Priority 1: Protect Network Backbone

Facilities:
- IND000000ACB
- IND562132AAA
- IND421302AAG

Actions:
- Capacity review
- Staffing redundancy
- Contingency routing

### Priority 2: Improve Critical Corridors

Focus on:
- IND000000ACB → IND562132AAA
- IND000000ACB → IND712311AAA
- IND000000ACB → IND501359AAE

Actions:
- Corridor diagnostics
- Schedule optimization
- Load balancing

### Priority 3: Reduce ETA Inflation Drivers

Primary operational drivers:
- Cutoff delays
- Scan delays
- Facility congestion
- Corridor congestion

Actions:
- Dispatch optimization
- Scan automation
- Dynamic load balancing

## Strategic Conclusion

The Delhivery network exhibits strong structural dependencies that are
not captured by traditional ETA systems.

Graph intelligence successfully captures these dependencies and should
be integrated into future ETA prediction and network planning systems.

The highest expected business value comes from:
1. Protecting critical hubs
2. Monitoring high-impact corridors
3. Deploying graph-aware ETA prediction
4. Building graph-powered operational dashboards

Facility priorities were assigned using impact-score percentiles.

Tier 1:
Top 20% of facilities by impact score.

Tier 2:
Facilities between the 60th and 80th percentile.

Tier 3:
Remaining facilities.

This approach provides a data-driven prioritization framework rather than an arbitrary ranking.

Critical:
Top 20% of corridors by impact score.

High:
60th–80th percentile.

Medium:
Remaining corridors.

In [111]:
strategy_memo_text="phase8_operations_strategy_memo.txt"
with open(
    f"{EXPORT_DIR}/phase8_operations_strategy_memo.txt",
    "w"
) as f:
    f.write(strategy_memo_text)

In [112]:
import os

root = "/content/drive/MyDrive/Delhivery_Graph_ETA"

for current_path, dirs, files in os.walk(root):
    level = current_path.replace(root, "").count(os.sep)

    if level > 3:
        continue

    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(current_path)}/")

    for f in files:
        print(f"{indent}    {f}")

Delhivery_Graph_ETA/
    data/
        feature_store/
            feature_store.csv
            model_dataset.csv
        graph_data/
            logistics_graph.gpickle
            connected_components.csv
            edge_features.csv
            graph_summary.csv
            logistics_graph.pkl
            node_features.csv
            top_corridors.csv
            top_hubs.csv
            node_centrality_features.csv
            community_features.csv
            graph_feature_store.csv
            graph_node_embeddings.csv
            graph_edge_features.csv
            graph_ml_feature_store.csv
        raw/
            delivery_data.csv
        clean/
            graph_ready_dataset.csv
    models/
        lgbm_final_graph_edge.pkl
        lgbm_final_graph_edge_features.json
        lgbm_final_graph_edge_metrics.json
        model_metadata.json
    reports/
        phase6_model_results.csv
        phase6_ablation_results.csv
        phase6_feature_importance.csv
        phase7_g

In [113]:
import pandas as pd

files = [
    "reports/phase6_model_results.csv",
    "reports/phase6_ablation_results.csv",
    "reports/phase7_graph_contribution_breakdown.csv",
    "reports/phase7_graph_family_contribution.csv",
    "reports/phase7_top_source_facilities.csv",
    "reports/phase7_top_destination_facilities.csv",
    "reports/phase7_top_impact_corridors.csv",
    "reports/phase8_hub_priorities.csv",
    "reports/phase8_corridor_priorities.csv"
]

base = "/content/drive/MyDrive/Delhivery_Graph_ETA/"

for f in files:
    print("\n" + "="*80)
    print(f)

    df = pd.read_csv(base + f)

    print(df.head())
    print("\nShape:", df.shape)


reports/phase6_model_results.csv
      Unnamed: 0       MAE      RMSE        R2       MAPE
0           Mean  0.623520  1.554625 -0.000227  29.746670
1    RF_Baseline  0.417790  0.981904  0.600989  19.286831
2       RF_Graph  0.412856  0.937229  0.636471  19.481117
3  LGBM_Baseline  0.447278  1.013578  0.574831  20.828476
4     LGBM_Graph  0.408220  0.932996  0.639748  18.817073

Shape: (5, 5)

reports/phase6_ablation_results.csv
                               Model       MAE      RMSE        R2       MAPE  \
0                           Baseline  0.444270  1.004123  0.582726  20.698032   
1              Baseline + Centrality  0.423169  0.959901  0.618671  19.726878   
2  Baseline + Centrality + Community  0.423188  0.956737  0.621180  19.626705   
3                         Full Graph  0.407613  0.930006  0.642053  18.800973   

   MAE_Gain_vs_Baseline  RMSE_Gain_vs_Baseline  
0              0.000000               0.000000  
1              4.749559               4.404003  
2            

In [114]:
import pandas as pd

files = [
    "reports/phase7_high_risk_corridors.csv",
    "reports/phase7_source_risk.csv",
    "reports/phase7_destination_risk.csv",
    "reports/phase7_corridor_risk.csv",
    "reports/phase7_executive_findings.csv",
    "reports/phase8_eta_reduction_actions.csv",

    "data/graph_data/graph_summary.csv",
    "data/graph_data/top_hubs.csv",
    "data/graph_data/top_corridors.csv",
    "data/graph_data/community_features.csv",
    "data/graph_data/node_centrality_features.csv",

    "models/lgbm_final_graph_edge_metrics.json",
    "models/model_metadata.json"
]

base = "/content/drive/MyDrive/Delhivery_Graph_ETA/"

for f in files:

    print("\n" + "="*100)
    print(f)

    if f.endswith(".csv"):
        df = pd.read_csv(base + f)

        print(df.head())
        print("\nColumns:")
        print(df.columns.tolist())
        print("\nShape:", df.shape)

    else:
        import json

        with open(base + f) as fp:
            obj = json.load(fp)

        print(obj)


reports/phase7_high_risk_corridors.csv
   Unnamed: 0                      corridor  shipments  avg_actual_factor  \
0           0  IND000000ACB -> IND562132AAA       1070           1.744211   
1           1  IND000000ACB -> IND712311AAA        597           2.435245   
2           2  IND000000ACB -> IND501359AAE        443           1.925686   
3           3  IND562132AAA -> IND000000ACB        460           1.785745   
4           4  IND000000ACB -> IND421302AAG        391           1.897166   

   avg_pred_factor  impact_score  
0         1.688481   1806.674239  
1         2.160609   1289.883763  
2         1.828349    809.958484  
3         1.698787    781.441933  
4         1.991502    778.677256  

Columns:
['Unnamed: 0', 'corridor', 'shipments', 'avg_actual_factor', 'avg_pred_factor', 'impact_score']

Shape: (20, 6)

reports/phase7_source_risk.csv
  source_center  shipments  avg_actual_factor  avg_pred_factor
0  IND110064AAA         55           3.967136         4.160798
1  IND4

In [118]:
import pandas as pd

df = pd.read_csv(
    "{REPORTS_DIR}/phase7_graph_family_contribution.csv"
)

print(df)


FileNotFoundError: [Errno 2] No such file or directory: '{REPORTS_DIR}/phase7_graph_family_contribution.csv'

In [ ]:
import pickle

with open(
    f"{GRAPH_DIR}/logistics_graph.pkl",
    "rb"
) as f:
    G = pickle.load(f)

print(type(G))

print(
    "Nodes:",
    G.number_of_nodes()
)

print(
    "Edges:",
    G.number_of_edges()
)

print(
    list(G.nodes(data=True))[:5]
)

print(
    list(G.edges(data=True))[:5]
)

In [ ]:
import pandas as pd

print(
    pd.read_csv(
        "{GRAPH_DIR}/top_hubs.csv"
    ).head()
)

print(
    pd.read_csv(
        "{GRAPH_DIR}/top_corridors.csv"
    ).head()
)